# 会议摘要器 —— 从会议记录生成会议纪要

## 练习目标（理念）

读取一份 **Word（.docx）** 会议记录，把它交给本地 **Ollama**（经 OpenAI 兼容接口），让模型生成清晰的 **会议纪要（Minutes of Meeting）**：要点、待办、决策。

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 文档读取 | `python-docx` 的 `Document` 逐段取文本 |
| system / user prompt | system 定「怎么写纪要」，user 塞入完整 transcript |
| OpenAI 兼容客户端 | `base_url` 指向本机 Ollama 的 `/v1` |
| 本地小模型 | `llama3.2:1b`（体积小、可离线试跑） |

## 怎么跑

1. 同目录准备好 `sample_meeting_transcript.docx`
2. 启动 Ollama，并确保已拉取 `llama3.2:1b`
3. 从上到下运行：读文档 → 写提示 → 建客户端 → 调用并打印摘要


In [5]:
# ========== 导入：文档读取 + OpenAI 兼容客户端 ==========

# 从 openai 导入 OpenAI：后面会把它指向本地 Ollama 的 OpenAI 兼容接口
from openai import OpenAI
# 从 python-docx 导入 Document：用来打开 .docx 并逐段读取会议记录正文
from docx import Document


In [6]:
# ========== 读取会议记录：把 .docx 全文拼成一个字符串 ==========

# 打开同目录下的示例会议记录文件（路径字符串保持原样，改路径会影响能否找到文件）
doc = Document('sample_meeting_transcript.docx')
# 遍历所有段落 paragraph，取出 .text，再用换行符拼成完整 transcript
content = '\n'.join([paragraph.text for paragraph in doc.paragraphs])
# 先打印原文，确认读取成功、内容是否完整
print(content)



Swati: Thanks everyone for joining. We've been seeing repeated errors in the application logs since yesterday. Rahul, can you summarize the issue?
Rahul: Sure. The logs show frequent 'Database connection timeout' errors coming from the Order Service, especially during peak traffic.
Neha: I checked the infrastructure metrics. CPU and memory look fine, but the database connection pool is getting exhausted.
Amit: From QA side, we noticed this happens when multiple users place orders simultaneously during load testing.
Swati: So the issue seems related to high concurrency. Rahul, why is the connection pool getting exhausted?
Rahul: The service is opening new DB connections but not closing them properly in one error-handling scenario.
Neha: That explains why connections keep increasing until the pool limit is reached.
Swati: So the root cause is improper connection handling in the Order Service code.
Amit: Yes, once the pool is exhausted, requests start failing, which matches what we obser

In [7]:
# ========== 写提示：system 定角色，user 塞入会议全文 ==========

# system_prompt：告诉模型「你是会议纪要专家」以及输出应包含要点 / 待办 / 决策
# 提示正文保持英文（影响模型行为的字符串不翻译）
system_prompt = """You are an expert in summarizing meeting transcripts.
Your task is to read the provided meeting transcript and generate a concise summary highlighting the key points discussed, action items, and decisions made during the meeting.
The summary should be clear, well-structured, and easy to understand."""

# user_prompt：用 f-string 把上一步读到的 content（会议全文）嵌进用户消息
user_prompt = f"""Provide the minutes of meeting for the meeting. Here is the meeting transcript: {content}"""


In [8]:
# ========== 连接本地 Ollama（OpenAI 兼容 /v1 接口）==========

# base_url 指向本机 Ollama 的 OpenAI 兼容端点；api_key 本地常可填任意非空占位（这里用 "llama"）
llama = OpenAI(base_url="http://localhost:11434/v1", api_key="llama")
# 选用的本地模型 id：须与 ollama list 里已安装的名字一致
MODEL="llama3.2:1b"


In [9]:
# ========== 调用本地模型：生成会议纪要并打印 ==========

# 通过 OpenAI 兼容接口发起 chat.completions；messages 里放 system + user
response = llama.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
)
# 取出第一条候选回复的文本内容并打印到笔记本输出
print(response.choices[0].message.content)


**Meeting Transcript Minutes**

**Attendees:** Swati, Rahul, Neha, Amit

**Topic:** Application Log Issues during Peak Traffic

**Summary:**

The meeting discussed repeated errors in application logs attributed to frequent 'Database connection timeout' errors from the Order Service, especially during peak traffic. The issue was identified through CPU and memory metrics checks by Neha and QA tests by Amit, indicating a high concurrency theme.

**Key Points:**

* Rahul reported that the error occurred due to improper connection handling in the Order Service code.
* Neha suggested updating the code to ensure proper connection closing and adding monitoring logs for earlier alerts on potential spikes in connection usage.
* Swati instructed Rahul to fix the code, with responsibility for completion handed to him for tomorrow.
* Amit agreed with Neha's suggestion for adjusting alerts to notify them sooner of increasing connection usage.

**Action Items:**

* Rahul: Fix database connection hand